# 🎙️ intiVoice AI — Studio Suara Pintar (Versi Resmi: V1.5.1)

> ⏰ **Terakhir Diperbarui:** 18 September 2026, 12:50 WIB | **Rilis:** V1.5.1 (Dynamic Voice Anchor & Dual-Speaker Parity)
> 🛡️ **Anti-Timeout Cloudflare:** Menggunakan Asynchronous Job Queue (`/job/create` & `/job/status`)

Notebook ini berfungsi sebagai **Engine Backend GPU Serverless** bertenaga Tesla T4 untuk aplikasi **intiVoice Web Studio**.

### ✨ Fitur Utama V1.5.0 (Dual-Speaker Dialogue & Modal Parity):
1. 👥 **Dual-Speaker Dialogue Engine**: Sintesis percakapan dua pembicara berbeda dengan kloning suara masing-masing, jeda santai alami, dan subtitle dual-speaker otomatis.
1. 🔒 **Voice Anchor DNA System**: Mengunci konsistensi timbre dan warna suara di seluruh bagian naskah panjang tanpa pergeseran vokal.
2. 🗣️ **Human-Cadence Sentence Chunking**: Naskah dipotong secara alami (180–240 karakter per kalimat) sehingga intonasi kalimat utuh dan tidak terburu-buru.
3. 🌬️ **Adaptive Breath Pause**: Jeda bernafas manusiawi antar kalimat (titik: 0.50s, koma: 0.25s, alinea: 0.70s) yang terhubung presisi ke subtitle.
4. ⚡ **Native Speed Control**: Kontrol kecepatan bicara alami via prompt instruction (bebas dari distorsi kaleng/metalik phase-vocoder).
5. 🛡️ **Torchaudio Smart Sinc Resampler**: Preprocessing audio referensi 16kHz mono dengan zero-degradation check bebas error `resampy`.
6. 🎙️ **Ultimate Voice Cloning**: Kloning vokal 1:1 via audio continuation + transcript (mereproduksi setiap desah nafas dan artikulasi vokal).
7. 🎛️ **Controllable Voice Cloning**: Kloning warna vokal (timbre) dari berkas audio referensi dengan style guidance.
8. 💾 **Persistent Google Drive Cache**: Bobot model disimpan permanen di Drive. Sekali unduh, sesi berikutnya 100% instan!
9. 🎬 **Dual Subtitle CapCut Generator**: Otomatis membuat .SRT & .ASS Karaoke (Mobile 9:16 safe zone).
10. 🌐 **Cloudflare Permanent Tunnel**: Mendukung URL tetap permanen via Cloudflare Tunnel Token.

---
### 🚀 Cara Menjalankan:
1. Klik tombol **Play (▶)** di sel kode di bawah ini.
2. Berikan izin koneksi Google Drive saat pop-up muncul agar cache model tersimpan permanen.
3. (Opsional) Masukkan **Cloudflare Tunnel Token** jika ingin URL selalu tetap, atau biarkan kosong untuk Quick Tunnel acak.
4. Tunggu hingga URL Cloudflare Tunnel muncul.
5. Salin URL tersebut ke aplikasi Web Studio pada **Tab Pengaturan Mesin**.


In [ ]:
"""
================================================================================
🎙️ INTIVOICE AI — ENGINE GPU & CLOUDFLARE TUNNEL (V1.4.1 STUDIO AUDIO ENGINE)
================================================================================
🚀 FITUR TERBARU V1.4.1:
   1. ⚡ Blazing-Fast Local SSD Cache: Model 4.6GB di-download langsung ke SSD lokal Colab.
      (Kecepatan tinggi NVMe tanpa bottleneck FUSE Google Drive & hemat 4.6GB kuota Drive!).
   2. 🎙️ Ultimate Voice Cloning: Kloning vokal 1:1 via audio continuation + transcript.
   3. 🎛️ Controllable Voice Cloning: Kloning timbre audio referensi dengan style guidance.
   4. 🔒 Voice Anchor DNA System: Mengunci konsistensi timbre vokal di seluruh bagian naskah.
   5. 🗣️ Human-Cadence Sentence Chunking: Naskah dipotong alami (180-240 karakter) per kalimat.
   6. 🌬️ Adaptive Breath Pause: Jeda bernafas manusiawi (titik: 0.5s, koma: 0.25s, alinea: 0.7s).
   7. ⚡ Native Speed Control via Prompt: Tempo diatur alami tanpa distorsi phase-vocoder.
   8. 🛡️ Torchaudio Sinc Resampler: Zero-degradation preprocessing 16kHz bebas error resampy.
   9. 🎬 Dual Subtitle CapCut Generator: Auto-export .SRT & .ASS Karaoke sinkron jeda nafas.
   10. 🌐 Cloudflare Permanent Named Tunnel: Dukungan URL tetap via Tunnel Token.
================================================================================
"""

# @title ⚙️ KONFIGURASI LISENSI TUNNEL & HUGGING FACE TOKEN (OPSIONAL)
# @markdown Masukkan kredensial lisensi atau Token Hugging Face untuk kecepatan download maksimal.
# @markdown *(Jika dikosongkan, mesin otomatis mendeteksi konfigurasi dari Google Drive `.env` atau Secrets).*

CLOUDFLARE_TUNNEL_TOKEN = ""  # @param {type:"string"}
CLOUDFLARE_TUNNEL_DOMAIN = ""  # @param {type:"string"}
HF_TOKEN = ""  # @param {type:"string"}

import os
import sys
import time
import subprocess
import threading
import re
import io
import random
import base64
import shutil
import tempfile
from typing import Optional, List, Dict, Any
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# ------------------------------------------------------------------------------
# 1. VERIFIKASI HARDWARE GPU TESLA T4
# ------------------------------------------------------------------------------
print("=" * 80)
print("🚀 MEMULAI INTIVOICE AI — ENGINE GPU & CLOUDFLARE TUNNEL (V1.4.1)")
print("🕒 Update: 18 September 2026, 10:15 WIB | 🎭 Stateful Emotion & Full Parity Active")
print("=" * 80)

print("\n[1/6] 🔍 Memeriksa Akselerator Hardware GPU...")
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)
        print(f"   ✅ GPU Terdeteksi: {gpu_name} ({vram_total:.0f} MB VRAM)")
    else:
        print("   ⚠️ PERINGATAN: GPU Tidak Terdeteksi! Model akan berjalan lambat di CPU.")
        print("   👉 Klik Runtime -> Change runtime type -> Pilih T4 GPU lalu Restart!")
except Exception as e:
    print(f"   ⚠️ Gagal memeriksa GPU: {e}")

# ------------------------------------------------------------------------------
# 2. INTEGRASI GOOGLE DRIVE PRIBADI (BACKUP AUDIO & LISENSI) + SSD MODEL CACHE
# ------------------------------------------------------------------------------
print("\n[2/6] 📁 Menghubungkan Google Drive untuk Backup Audio & Lisensi...")
DRIVE_MOUNTED = False
DRIVE_APP_DIR = "/content/drive/MyDrive/intiVoice_Audio"
DRIVE_OUTPUT = DRIVE_APP_DIR

# HF_CACHE diset ke NVMe SSD lokal Colab agar loading secepat kilat (bebas bottleneck FUSE)
# dan kuota 15 GB Google Drive pengguna 100% hemat untuk audio!
HF_CACHE = "/root/.cache/huggingface"
os.environ['HF_HOME'] = HF_CACHE
os.environ['HF_HUB_CACHE'] = HF_CACHE
os.makedirs(HF_CACHE, exist_ok=True)

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("   ⏳ Menghubungkan ke Google Drive pribadi Anda...")
        drive.mount('/content/drive')
    DRIVE_MOUNTED = True
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)
    print(f"   ✅ Google Drive Terhubung!")
    print(f"   📂 Output Hasil Audio   : {DRIVE_OUTPUT}")
    print(f"   ⚡ Model Cache (SSD Colab): {HF_CACHE} (Hemat 4.6GB Kuota Google Drive)")
except Exception as e:
    print(f"   ℹ️ Google Drive tidak terhubung ({e}). Menggunakan disk lokal sementara...")
    DRIVE_OUTPUT = "/content/outputs"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# ------------------------------------------------------------------------------
# 3. INSTALASI DEPENDENSI ULTRA-FAST VIA UV
# ------------------------------------------------------------------------------
print("\n[3/6] ⚡ Memeriksa & memasang dependensi (Ultra-Fast via uv)...")
def install_fast_dependencies():
    try:
        import uv
    except ImportError:
        print("   ⏳ Memasang paket manager uv (kecepatan 10x pip)...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv"])
    
    try:
        import voxcpm
        import soundfile
        import librosa
        import fastapi
        import uvicorn
        import pydantic
        import resampy
        import soxr
        import torchaudio
        print("   ✅ Seluruh pustaka audio & AI sudah terpasang!")
    except ImportError:
        print("   ⏳ Memasang dependensi inti...")
        subprocess.check_call([
            "uv", "pip", "install", "--system",
            "voxcpm", "soundfile", "librosa", "gradio", "fastapi",
            "uvicorn", "pydantic", "resampy", "soxr", "torchaudio"
        ])
        print("   ✅ Dependensi berhasil dipasang!")

install_fast_dependencies()

import numpy as np
import soundfile as sf
import librosa
import torchaudio
from voxcpm import VoxCPM
from fastapi import FastAPI, Response, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# ------------------------------------------------------------------------------
# 4. MEMASANG CLOUDFLARED (TUNNEL ENGINE)
# ------------------------------------------------------------------------------
print("\n[4/6] 🌐 Menyiapkan Cloudflare Tunnel...")
if not os.path.exists("/usr/local/bin/cloudflared"):
    print("   ⏳ Mengunduh binary cloudflared resmi...")
    subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"])
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])
    print("   ✅ Cloudflared siap digunakan!")
else:
    print("   ✅ Cloudflared sudah terpasang.")

# ------------------------------------------------------------------------------
# 5. INISIALISASI MODEL KE GPU (ULTRA-FAST NVMe SSD STREAMING)
# ------------------------------------------------------------------------------
print("\n[5/6] 🧠 Memuat Model Suara AI ke Memori GPU...")
if 'model' not in globals():
    t_start = time.time()
    print("   ⚡ Mengunduh/memuat bobot model via NVMe SSD Colab (~4.6GB)...")
    print("   🚀 Kecepatan transfer Gigabit Datacenter (bebas hambatan FUSE Drive)...")

    model = VoxCPM.from_pretrained(
        'openbmb/VoxCPM2',
        cache_dir=HF_CACHE,
        load_denoiser=False
    )
    
    load_duration = time.time() - t_start
    vram_used = torch.cuda.memory_allocated(0) / (1024 ** 2) if torch.cuda.is_available() else 0
    print(f"   ✅ MODEL SUARA AI SIAP DI GPU! (⏱️ Waktu Load: {load_duration:.1f} detik) | VRAM: {vram_used:.1f} MB")
else:
    vram_used = torch.cuda.memory_allocated(0) / (1024 ** 2) if torch.cuda.is_available() else 0
    print(f"   ⚡ RE-ATTACH BERHASIL: Model sudah aktif di GPU VRAM ({vram_used:.1f} MB) tanpa reload!")

# ------------------------------------------------------------------------------
# 6. HELPER DSP & RITME NAFAS ALAMI (HUMAN CADENCE & BREATH PAUSE)
# ------------------------------------------------------------------------------
def apply_fade(audio: np.ndarray, fade_samples: int = 480) -> np.ndarray:
    """Terapkan envelope fade 10ms di batas chunk untuk menghilangkan bunyi klik/pop."""
    result = audio.copy()
    fs = min(fade_samples, len(result) // 4)
    if fs > 0:
        fade = np.linspace(0.0, 1.0, fs, dtype=np.float32)
        result[:fs] *= fade
        result[-fs:] *= fade[::-1]
    return result

def speed_to_control_hint(speed: float) -> str:
    """Konversi faktor kecepatan numerik ke instruksi teks native model VoxCPM2."""
    speed_map = [
        (0.50, 0.70, "very slowly and clearly"),
        (0.70, 0.85, "slowly"),
        (0.85, 0.92, "slightly slower"),
        (0.92, 1.08, ""),
        (1.08, 1.20, "slightly faster"),
        (1.20, 1.40, "fast"),
        (1.40, 2.01, "very fast"),
    ]
    for lo, hi, hint in speed_map:
        if lo <= speed < hi:
            return hint
    return ""

def get_adaptive_pause_sec(text: str) -> float:
    """Hitung jeda bernafas adaptif manusiawi berdasarkan tanda baca akhir klausa/kalimat."""
    t = text.strip()
    if not t:
        return 0.35
    last_char = t[-1]
    if last_char in ['.', '!', '?', '。', '！', '？']:
        return 0.50
    elif last_char in [',', ';', ':', '-', '—', '，', '、']:
        return 0.25
    elif '\n' in text:
        return 0.70
    return 0.40

def split_text_into_smart_chunks(text: str, max_chars_per_chunk: int = 240) -> List[str]:
    """Smart sentence splitter yang mematuhi ritme nafas manusiawi (Human Cadence 240 karakter)."""
    clean_text = text.replace('\r\n', '\n').replace('\r', '\n').strip()
    if len(clean_text) <= max_chars_per_chunk:
        return [clean_text] if clean_text else []

    sentence_pattern = r'(?<=[.!?。！？\n])\s+'
    raw_sentences = [s.strip() for s in re.split(sentence_pattern, clean_text) if s.strip()]

    chunks = []
    current_chunk = ""

    for s in raw_sentences:
        if len(current_chunk) + len(s) + 1 <= max_chars_per_chunk:
            current_chunk = f"{current_chunk} {s}".strip()
        else:
            if current_chunk:
                chunks.append(current_chunk)
            if len(s) > max_chars_per_chunk:
                clause_pattern = r'(?<=[,;:\-—，、])\s+'
                clauses = [c.strip() for c in re.split(clause_pattern, s) if c.strip()]
                sub_chunk = ""
                for c in clauses:
                    if len(sub_chunk) + len(c) + 1 <= max_chars_per_chunk:
                        sub_chunk = f"{sub_chunk} {c}".strip()
                    else:
                        if sub_chunk:
                            chunks.append(sub_chunk)
                        sub_chunk = c
                if sub_chunk:
                    chunks.append(sub_chunk)
                current_chunk = ""
            else:
                current_chunk = s

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

def format_timestamp_srt(seconds: float) -> str:
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    millis = int(round((seconds - int(seconds)) * 1000))
    if millis >= 1000:
        millis = 999
    return f"{hrs:02d}:{mins:02d}:{secs:02d},{millis:03d}"

def format_timestamp_ass(seconds: float) -> str:
    hrs = int(seconds // 3600)
    mins = int((seconds % 3600) // 60)
    secs = int(seconds % 60)
    centis = int(round((seconds - int(seconds)) * 100))
    if centis >= 100:
        centis = 99
    return f"{hrs:01d}:{mins:02d}:{secs:02d}.{centis:02d}"

EMOTION_THESAURUS = {
    "sedih": "sad and emotional voice, melancholic trembling tone, slow pace",
    "menangis": "crying, sobbing, tearful, shaky and weak voice",
    "marah": "angry and dramatic tone, intense forceful projection, tense pace, aggressive",
    "senang": "happy, cheerful, energetic, bright tone",
    "gembira": "happy, cheerful, energetic, bright tone",
    "bahagia": "happy, joyful, warm, smiling tone",
    "ceria": "cheerful and laughing tone, joyful smiling voice, bright and lively",
    "tertawa": "cheerful and laughing tone, joyful smiling voice, bright and lively",
    "antusias": "enthusiastic and energetic tone, high pitch, excited delivery",
    "semangat": "enthusiastic and energetic tone, high pitch, excited delivery",
    "bisik": "whispering softly, breathy intimate voice, low volume",
    "berbisik": "whispering softly, breathy intimate voice, low volume",
    "takut": "scared, terrified, panicked, trembling, breathless",
    "panik": "panicked, rushed, breathless, agitated tone",
    "kaget": "surprised, shocked, gasped",
    "terkejut": "surprised, shocked, gasped",
    "tegas": "deep authoritative tone, strong projection, firm and confident",
    "wibawa": "deep authoritative tone, strong projection, firm and confident",
    "lelah": "tired, exhausted, slow and faint voice",
    "lemas": "tired, exhausted, slow and faint voice",
    "sombong": "sarcastic, arrogant, smirking tone",
    "sinis": "sarcastic, cynical, sneering tone",
    "romantis": "warm romantic voice, soft intimate tone, slow tender pacing",
    "tenang": "calm and relaxed tone, soothing gentle pitch, slow pacing",
}

def parse_dialogue_chunk(text: str):
    raw = text.strip()
    match = re.match(r'^(?:([^:()]+):)?\s*(?:\(([^)]+)\))?\s*(.*)$', raw, re.DOTALL)
    if not match:
        return None, None, raw
    speaker = match.group(1).strip() if match.group(1) else None
    emotion = match.group(2).strip() if match.group(2) else None
    spoken = match.group(3).strip() if match.group(3) else raw
    spoken = re.sub(r'\([^)]*\)', '', spoken).strip()
    return speaker, emotion, spoken

# ------------------------------------------------------------------------------
# 7. FASTAPI BACKEND API & KONTRAK DATA
# ------------------------------------------------------------------------------
print("\n[6/6] 🚀 Menyiapkan Smart Auto-Chunker & Backend API...")

api_app = FastAPI(title="intiVoice Engine Colab")
api_app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)


# ------------------------------------------------------------------------------
# DUAL-SPEAKER DIALOGUE MODELS & HELPERS (v1.5.0)
# ------------------------------------------------------------------------------
class DialogueTurn(BaseModel):
    speaker: str
    emotion_tag: Optional[str] = None
    text: str
    pause_after: float = 0.40

class SpeakerProfile(BaseModel):
    name: str
    reference_audio_b64: Optional[str] = None
    prompt_text: Optional[str] = None
    style_prompt: Optional[str] = None
    seed: Optional[int] = 42
    speed: Optional[float] = 1.0
    avatar: Optional[str] = None

class DialogueGenerateRequest(BaseModel):
    script: Optional[str] = None
    speakers: Dict[str, SpeakerProfile]
    turns: Optional[List[DialogueTurn]] = None
    cfg_value: Optional[float] = 2.0
    inference_timesteps: Optional[int] = 10
    default_pause_sec: Optional[float] = 0.40
    audio_id: Optional[str] = None

class TTSRequest(BaseModel):
    text: str
    prompt_voice: Optional[str] = ""
    reference_audio_b64: Optional[str] = None
    prompt_text: Optional[str] = None
    refTranscript: Optional[str] = None
    cfg_value: Optional[float] = 2.0
    temperature: Optional[float] = 2.0
    inference_timesteps: Optional[int] = 10
    speed: Optional[float] = 1.0
    seed: Optional[int] = None
    audio_id: Optional[str] = None

@api_app.get("/")
@api_app.get("/health")
def root():
    return {
        "status": "online",
        "ready": True,
        "ok": True,
        "engine": "intiVoice AI Engine (Colab T4 Studio)",
        "version": "1.5.0",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
        "cloning_supported": True,
        "cloning_modes": ["controllable", "ultimate"],
        "sample_rate": model.tts_model.sample_rate if 'model' in globals() else 48000,
        "audio_quality_fixes": [
            "no-phase-vocoder",
            "smart-resample",
            "chunk-fade",
            "random-seed",
            "voice-dna-anchor",
            "native-speed-control",
            "retry-badcase",
            "realtime-chunk-progress",
            "human-cadence-chunking",
            "natural-breathing-pauses",
            "prompt-isolation",
            "emotive-voice-cloning",
            "audio-vault-drive",
            "dialogue-emotion-parser",
        ],
    }


# ------------------------------------------------------------------------------
# PYDANTIC MODELS TAMBAHAN UNTUK PARITAS API V1 & OPENAI
# ------------------------------------------------------------------------------
class DesignRequest(BaseModel):
    text: str
    voice_description: str = ""
    cfg_value: float = 2.0
    inference_timesteps: int = 10
    seed: Optional[int] = None
    speed: float = 1.0

class CloneRequest(BaseModel):
    text: str
    reference_audio_b64: Optional[str] = None
    prompt_text: Optional[str] = None
    cfg_value: float = 2.0
    inference_timesteps: int = 10
    seed: Optional[int] = None
    speed: float = 1.0

class OpenAIRequest(BaseModel):
    model: str = "voxcpm2"
    input: str
    voice: Optional[str] = None
    speed: Optional[float] = 1.0
    response_format: Optional[str] = "wav"

def purge_expired_files(directory_path: str, max_age_seconds: int = 7200, allowed_extensions=(".wav", ".srt", ".ass", ".txt", ".json")):
    """Memindai dan menghapus berkas audio keluaran yang berusia > 2 jam (7200s)."""
    stats = {"scanned": 0, "deleted": 0, "freed_bytes": 0}
    if not directory_path or not os.path.exists(directory_path) or not os.path.isdir(directory_path):
        return stats
    now = time.time()
    try:
        for entry in os.listdir(directory_path):
            file_path = os.path.join(directory_path, entry)
            if not os.path.isfile(file_path):
                continue
            _, ext = os.path.splitext(entry)
            if ext.lower() not in allowed_extensions:
                continue
            stats["scanned"] += 1
            try:
                st = os.stat(file_path)
                if (now - st.st_mtime) >= max_age_seconds:
                    os.unlink(file_path)
                    stats["deleted"] += 1
                    stats["freed_bytes"] += st.st_size
            except Exception:
                pass
    except Exception:
        pass
    return stats

def _start_colab_storage_cleanup_daemon():
    """Background daemon: memindai dan menghapus berkas audio kedaluwarsa (> 2 jam) setiap 30 menit."""
    def _loop():
        while True:
            time.sleep(1800)
            try:
                target_dir = globals().get('DRIVE_OUTPUT')
                if target_dir and os.path.exists(target_dir):
                    purge_expired_files(target_dir, max_age_seconds=7200)
                if os.path.exists('/content/outputs') and '/content/outputs' != target_dir:
                    purge_expired_files('/content/outputs', max_age_seconds=7200)
            except Exception:
                pass
    t = threading.Thread(target=_loop, daemon=True)
    t.start()

_start_colab_storage_cleanup_daemon()

def cleanup_expired_jobs(max_age_sec: float = 1800.0):
    """Bersihkan job-job lama dari RAM Colab agar memori tidak bocor."""
    now = time.time()
    expired_keys = [k for k, v in jobs.items() if (now - v.get("created_at", now)) > max_age_sec]
    for k in expired_keys:
        try:
            del jobs[k]
        except KeyError:
            pass

    try:
        target_dir = globals().get('DRIVE_OUTPUT')
        if target_dir and os.path.exists(target_dir):
            purge_expired_files(target_dir, max_age_seconds=7200)
    except Exception:
        pass
def resolve_stateful_emotion(active_emotion_state: str, local_emotion_tag: Optional[str], global_control_hint: str = ""):
    """Stateful Emotion Memory: mewariskan emosi aktif ke potongan kalimat berikutnya."""
    if local_emotion_tag and local_emotion_tag.strip():
        tag_clean = local_emotion_tag.strip().lower()
        translated = EMOTION_THESAURUS.get(tag_clean, local_emotion_tag.strip())
        return translated, translated
    if active_emotion_state and active_emotion_state.strip():
        return active_emotion_state, active_emotion_state
    return global_control_hint, global_control_hint


# ------------------------------------------------------------------------------
# 8. ENGINE SINTESIS SUARA DENGAN VOICE ANCHOR & JEDA NAFAS ALAMI
# ------------------------------------------------------------------------------
jobs = {}

def _internal_synthesize(req: TTSRequest, progress_fn=None) -> dict:
    if not req.text or not req.text.strip():
        raise HTTPException(status_code=400, detail="Teks naskah tidak boleh kosong!")
    
    temp_files = []
    try:
        total_len = len(req.text)
        
        # 1. Kunci seed acak natural per-request (prosodi hidup, tidak monoton)
        target_seed = int(req.seed) if (req.seed is not None and req.seed >= 0) else random.randint(1000, 999999)
        torch.manual_seed(target_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(target_seed)
        np.random.seed(target_seed)
        random.seed(target_seed)

        # 2. CFG Guidance Scale
        target_cfg = 2.0
        if req.cfg_value is not None and float(req.cfg_value) >= 1.0:
            target_cfg = float(req.cfg_value)
        elif req.temperature is not None and float(req.temperature) >= 1.0:
            target_cfg = float(req.temperature)

        # 3. Kontrol Kecepatan Bicara Native & Ekstraksi Emosi
        target_speed = float(req.speed) if req.speed is not None else 1.0
        speed_hint = speed_to_control_hint(target_speed)

        emotion_match = re.match(r'^\s*\(([^)]+)\)\s*', req.text)
        raw_emotion_tag = emotion_match.group(1).strip() if emotion_match else ""

        style_parts = []
        if speed_hint:
            style_parts.append(speed_hint)
        is_cloning_requested = bool(req.reference_audio_b64 and req.reference_audio_b64.strip())
# 2. Tag Emosi Cepat (Prioritas Vokal Tertinggi via EMOTION_THESAURUS)
        clean_emotion = ""
        if raw_emotion_tag:
            clean_emotion = EMOTION_THESAURUS.get(raw_emotion_tag.lower(), raw_emotion_tag)
            if clean_emotion:
                style_parts.append(clean_emotion)

        # 3. Deskripsi Karakter Suara
        if req.prompt_voice and (not is_cloning_requested or not clean_emotion):
            clean_voice = re.sub(r"[()（）]", "", req.prompt_voice).strip()
            if clean_voice:
                style_parts.append(clean_voice)

        # 4. Panduan ritme nafas tenang default (jika tanpa emosi khusus & tanpa speed khusus)
        if not speed_hint and not clean_emotion:
            style_parts.insert(0, "relaxed pace, natural pauses between sentences")

        clean_control = ", ".join(style_parts)

        base_text = re.sub(r'^\s*\([^)]+\)\s*', '', req.text).strip()
        if not base_text:
            base_text = req.text.strip()

        # 4. Tangani Audio Referensi Kloning via Torchaudio Smart Resampler
        ref_wav_path = None
        prompt_text_clean = None
        is_ultimate_cloning = False
        prompt_txt_input = req.prompt_text or req.refTranscript

        if req.reference_audio_b64 and req.reference_audio_b64.strip():
            try:
                audio_bytes = base64.b64decode(req.reference_audio_b64)
                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_in:
                    tmp_in.write(audio_bytes)
                    raw_audio_path = tmp_in.name
                    temp_files.append(raw_audio_path)

                ref_info = sf.info(raw_audio_path)
                with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_16k:
                    ref_wav_path = tmp_16k.name
                    temp_files.append(ref_wav_path)

                # Resample ke 16kHz Mono hanya jika format berbeda (zero degradation)
                if ref_info.samplerate != 16000 or ref_info.channels > 1:
                    waveform, orig_sr = torchaudio.load(raw_audio_path)
                    if waveform.shape[0] > 1:
                        waveform = torch.mean(waveform, dim=0, keepdim=True)
                    if orig_sr != 16000:
                        resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=16000)
                        waveform = resampler(waveform)
                    torchaudio.save(ref_wav_path, waveform, 16000)
                else:
                    shutil.copy(raw_audio_path, ref_wav_path)

                if prompt_txt_input and prompt_txt_input.strip():
                    prompt_text_clean = prompt_txt_input.strip()
                    is_ultimate_cloning = True
                    # Emotive Cloning: clean_control dipertahankan agar tag emosi aktif
                    print(f"   🎙️ [Ultimate Voice Cloning] Aktif! Meniru nuansa vokal mikro 1:1 ({len(prompt_text_clean)} karakter transkrip)")
                else:
                    print(f"   🎛️ [Controllable Voice Cloning] Aktif! Meniru timbre vokal dasar pembicara")
            except Exception as clone_err:
                print(f"   ⚠️ Gagal memproses audio kloning: {clone_err}. Fallback ke Voice Design.")
                ref_wav_path = None
                prompt_text_clean = None

        # 5. Smart Sentence Chunking (Human Cadence: 240 karakter per kalimat alami)
        chunks = split_text_into_smart_chunks(base_text, max_chars_per_chunk=240)
        num_chunks = len(chunks)
        sample_rate = model.tts_model.sample_rate

        mode_desc = "Ultimate Clone" if is_ultimate_cloning else ("Timbre Clone" if ref_wav_path else "Voice Design")
        print("-" * 80)
        print(f"🎮 [GPU T4 Processing] Mode: {mode_desc} | Naskah: {total_len} karakter | {num_chunks} Bagian | Seed: #{target_seed} | CFG: {target_cfg}")
        if clean_control:
            print(f"   🎛️ Control Instruction: '({clean_control})'")

        audio_segments = []
        chunk_durations = []

        if progress_fn:
            progress_fn(f"Menyiapkan {num_chunks} bagian naskah di GPU...")

        # 6. VOICE DNA ANCHOR SYSTEM (Kunci konsistensi warna suara naskah panjang)
        voice_anchor_path = None
        use_voice_anchor = (ref_wav_path is None) and (num_chunks > 1)
        sub_chunks = []
        active_emotion_state = clean_emotion  # Stateful Emotion Memory Tracker

        for i, chunk in enumerate(chunks):
            clean_chunk = chunk.replace('\n', ' ').strip()
            clean_chunk = re.sub(r'\s+', ' ', clean_chunk)
            if not clean_chunk:
                continue

            speaker, emotion_tag, clean_spoken = parse_dialogue_chunk(clean_chunk)
            if not clean_spoken:
                clean_spoken = clean_chunk

            sub_text = f"{speaker}: {clean_spoken}" if speaker else clean_spoken
            sub_chunks.append(sub_text)

            # Stateful Emotion Memory: pertahankan emosi aktif ke kalimat berikutnya
            active_emotion_state, chunk_control = resolve_stateful_emotion(
                active_emotion_state, emotion_tag, clean_control
            )

            if speed_hint:
                chunk_control = f"{chunk_control}, {speed_hint}".strip(", ")

            t_chunk = time.time()

            chunk_seed = target_seed + i
            torch.manual_seed(chunk_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(chunk_seed)
            np.random.seed(chunk_seed)

            prompt = f"({chunk_control}){clean_spoken}" if chunk_control else clean_spoken
            gen_kwargs = {
                "text": prompt,
                "cfg_value": target_cfg,
                "inference_timesteps": req.inference_timesteps or 10,
                "retry_badcase": True,
                "retry_badcase_max_times": 2,
            }

            active_ref_path = ref_wav_path or voice_anchor_path
            if active_ref_path:
                gen_kwargs["reference_wav_path"] = active_ref_path
                # Prompt Isolation: Dilarang kirim prompt_wav_path/prompt_text ke chunk

            with torch.inference_mode():
                wav = model.generate(**gen_kwargs)

            # Buat Voice Anchor dari 3 detik pertama chunk-0 untuk mengunci DNA suara berikutnya
            if i == 0 and use_voice_anchor and voice_anchor_path is None:
                try:
                    anchor_file = tempfile.NamedTemporaryFile(suffix="_anchor.wav", delete=False)
                    anchor_wav = wav[:int(sample_rate * 3)] if len(wav) > int(sample_rate * 3) else wav
                    sf.write(anchor_file.name, anchor_wav, sample_rate)
                    voice_anchor_path = anchor_file.name
                    temp_files.append(voice_anchor_path)
                    print(f"   🔒 Voice Anchor aktif ({len(anchor_wav)/sample_rate:.1f}s) → Mengunci DNA vokal untuk {num_chunks-1} bagian berikutnya")
                except Exception as anchor_err:
                    print(f"   ⚠️ Gagal membuat Voice Anchor: {anchor_err}")
                    voice_anchor_path = None

            # Micro-Fade 10ms pada sambungan audio
            faded_wav = apply_fade(wav, fade_samples=int(sample_rate * 0.01))
            audio_segments.append(faded_wav)
            dur_chunk = len(wav) / sample_rate
            chunk_durations.append(dur_chunk)

            # Adaptive Breath Pause System (Jeda hening bernafas adaptif)
            pause_sec = get_adaptive_pause_sec(clean_chunk)
            silence_pause = np.zeros(int(sample_rate * pause_sec), dtype=np.float32)
            if i < num_chunks - 1:
                audio_segments.append(silence_pause)

            proc_time = time.time() - t_chunk
            print(f"   ✅ Bagian [{i+1}/{num_chunks}] | Durasi: {dur_chunk:.2f}s | Jeda: {pause_sec:.2f}s | Waktu: {proc_time:.2f}s | {'🔒 Anchor' if active_ref_path else '🎲 Free'}")
            if progress_fn:
                progress_fn(f"Bagian [{i+1}/{num_chunks}] selesai ({dur_chunk:.1f}s)")

        full_audio = np.concatenate(audio_segments) if len(audio_segments) > 1 else audio_segments[0]
        final_duration = len(full_audio) / sample_rate
        print(f"🎉 SUKSES: Durasi Akhir {final_duration:.2f}s | Sample Rate: {sample_rate}Hz")

        # 7. SIMPAN OTOMATIS 4-BERKAS KE GOOGLE DRIVE
        random_id = f"{random.randint(1000, 9999)}"
        timestamp_str = time.strftime('%Y%m%d_%H%M%S')
        clean_name = (req.audio_id or f"audio_{timestamp_str}_{random_id}").replace(".wav", "")
        base_name = clean_name
        
        save_wav_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.wav")
        save_txt_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.txt")
        save_srt_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.srt")
        save_ass_path = os.path.join(DRIVE_OUTPUT, f"{base_name}.ass")

        # Subtitle SRT & ASS Generator dengan Sinkronisasi Jeda Adaptif
        scale_ratio = 1.0
        srt_lines = []
        curr_time = 0.0

        subtitle_list = sub_chunks if sub_chunks else chunks
        for idx, (c_text, dur) in enumerate(zip(subtitle_list, chunk_durations)):
            seg_dur = dur * scale_ratio
            start_str = format_timestamp_srt(curr_time)
            end_str = format_timestamp_srt(curr_time + seg_dur)
            srt_lines.append(f"{idx + 1}\n{start_str} --> {end_str}\n{c_text.strip()}\n")
            seg_pause = get_adaptive_pause_sec(c_text) if idx < len(chunks) - 1 else 0.0
            curr_time += seg_dur + (seg_pause * scale_ratio)
        
        srt_content = "\n".join(srt_lines) + "\n"

        ass_header = """[Script Info]
Title: intiVoice Studio Audio Subtitles
ScriptType: v4.00+
WrapStyle: 0
ScaledBorderAndShadow: yes
YCbCr Matrix: None
PlayResX: 1080
PlayResY: 1920

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Montserrat,58,&H00FFFFFF,&H0000D2B4,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,3.5,1.5,2,70,70,280,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
        ass_events = []
        curr_time_ass = 0.0
        subtitle_list = sub_chunks if sub_chunks else chunks
        for idx, (c_text, dur) in enumerate(zip(subtitle_list, chunk_durations)):
            seg_dur = dur * scale_ratio
            start_ass = format_timestamp_ass(curr_time_ass)
            end_ass = format_timestamp_ass(curr_time_ass + seg_dur)
            clean_cap = c_text.strip().replace('\n', ' ')
            ass_events.append(f"Dialogue: 0,{start_ass},{end_ass},Default,,0,0,0,,{clean_cap}")
            seg_pause = get_adaptive_pause_sec(c_text) if idx < len(chunks) - 1 else 0.0
            curr_time_ass += seg_dur + (seg_pause * scale_ratio)

        ass_content = ass_header + "\n".join(ass_events) + "\n"

        try:
            sf.write(save_wav_path, full_audio, sample_rate, format='WAV')
            with open(save_txt_path, 'w', encoding='utf-8') as f:
                f.write(f"=== METADATA SINTESIS SUARA INTIVOICE AI ===\n")
                f.write(f"Waktu       : {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
                f.write(f"Karakter    : {req.prompt_voice or '-'}\n")
                f.write(f"Mode        : {mode_desc}\n")
                f.write(f"Seed DNA    : {target_seed}\n")
                f.write(f"CFG Scale   : {target_cfg}\n")
                f.write(f"Speed       : {target_speed}x\n")
                f.write(f"Durasi Audio: {final_duration:.2f} detik\n")
                f.write(f"Total Karakter: {total_len}\n")
                f.write(f"Jumlah Bagian : {num_chunks}\n\n")
                f.write(f"--- NASKAH TEKS ---\n{base_text}\n")
            
            with open(save_srt_path, 'w', encoding='utf-8') as f:
                f.write(srt_content)

            with open(save_ass_path, 'w', encoding='utf-8') as f:
                f.write(ass_content)

            print(f"   💾 Auto-Save 4-Berkas Berhasil:")
            print(f"      🎵 WAV : {save_wav_path}")
            print(f"      📄 TXT : {save_txt_path}")
            print(f"      🎬 SRT : {save_srt_path} (CapCut / Premiere Ready)")
            print(f"      ✨ ASS : {save_ass_path} (TikTok / Shorts Safe Zone 9:16)")
        except Exception as drive_err:
            print(f"   ⚠️ Gagal menyimpan arsip ke Drive: {drive_err}")

        out_buf = io.BytesIO()
        sf.write(out_buf, full_audio, sample_rate, format='WAV')
        out_buf.seek(0)
        wav_bytes = out_buf.read()
        b64_str = f"data:audio/wav;base64,{base64.b64encode(wav_bytes).decode('utf-8')}"

        return {
            "wav_bytes": wav_bytes,
            "data_url": b64_str,
            "final_duration": final_duration,
            "mode_desc": mode_desc,
            "base_name": base_name,
            "srt_content": srt_content,
            "ass_content": ass_content,
        }

    except Exception as e:
        print(f"❌ ERROR SINTESIS: {e}")
        import traceback
        traceback.print_exc()
        raise HTTPException(status_code=500, detail=str(e))
    finally:
        for f in temp_files:
            try:
                if os.path.exists(f):
                    os.unlink(f)
            except:
                pass

@api_app.post("/job/create")
def create_job(req: TTSRequest):
    cleanup_expired_jobs()
    if not req.text or not req.text.strip():
        raise HTTPException(status_code=400, detail="Teks naskah tidak boleh kosong!")
    
    job_id = f"job_{int(time.time()*1000)}_{random.randint(100, 999)}"
    jobs[job_id] = {
        "status": "processing",
        "progress": "Menyiapkan naskah di GPU...",
        "created_at": time.time(),
        "result": None,
        "error": None
    }
    
    def worker():
        try:
            def on_prog(msg):
                if job_id in jobs:
                    jobs[job_id]["progress"] = msg
            res = _internal_synthesize(req, progress_fn=on_prog)
            if job_id in jobs:
                jobs[job_id]["status"] = "completed"
                jobs[job_id]["progress"] = "Selesai!"
                jobs[job_id]["result"] = {
                    "audio_b64": res["data_url"],
                    "duration": res["final_duration"],
                    "mode": res["mode_desc"],
                    "filename": f"{res['base_name']}.wav",
                    "srt_content": res.get("srt_content", ""),
                    "ass_content": res.get("ass_content", ""),
                }
        except Exception as err:
            if job_id in jobs:
                jobs[job_id]["status"] = "failed"
                jobs[job_id]["error"] = str(err)
                print(f"❌ Error Async Job {job_id}: {err}")
    
    worker_t = threading.Thread(target=worker, daemon=True)
    worker_t.start()
    return {"job_id": job_id, "status": "processing"}

@api_app.get("/job/status/{job_id}")
def get_job_status(job_id: str):
    cleanup_expired_jobs()
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail="Job ID tidak ditemukan atau sudah kedaluwarsa")
    item = jobs[job_id]
    resp = {
        "job_id": job_id,
        "status": item["status"],
        "progress": item.get("progress", ""),
        "error": item.get("error")
    }
    if item["status"] == "completed" and item.get("result"):
        resp["result"] = item["result"]
    return resp

@api_app.post("/generate")
def generate_tts(req: TTSRequest):
    res = _internal_synthesize(req)
    return Response(
        content=res["wav_bytes"],
        media_type="audio/wav",
        headers={
            "Content-Disposition": f"attachment; filename={res['base_name']}.wav",
            "X-Audio-Duration": f"{res['final_duration']:.2f}",
            "X-Audio-Cloning-Mode": res["mode_desc"],
            "X-Engine-Version": "1.5.0",
            "X-Build-Time": time.strftime('%Y-%m-%d %H:%M:%S'),
        }
    )





# ------------------------------------------------------------------------------
# DUAL-SPEAKER DIALOGUE ENGINE CORE FUNCTIONS (v1.5.0)
# ------------------------------------------------------------------------------
def parse_dialogue_script(script: str, default_pause: float = 0.40) -> List[DialogueTurn]:
    """Mem-parse naskah percakapan bebas menjadi giliran bicara DialogueTurn."""
    turns: List[DialogueTurn] = []
    if not script or not script.strip():
        return turns

    clean_script = script.replace("\r\n", "\n").replace("\r", "\n")
    lines = clean_script.strip().split("\n")

    for line in lines:
        raw_line = line.strip()
        if not raw_line:
            continue

        line_clean = re.sub(r"^(?:\d+[\.\)]|\-|\*|•)\s*", "", raw_line).strip()
        if not line_clean:
            continue

        speaker, emotion_tag, content = None, None, None
        colon_idx = line_clean.find(":")
        bracket_match = re.match(r"^\[([^\]]+)\]\s*(?:\(([^)]+)\)|\[([^\]]+)\])?\s*:?\s*(.*)$", line_clean)

        if bracket_match:
            speaker = bracket_match.group(1).strip()
            emotion_tag = (bracket_match.group(2) or bracket_match.group(3) or "").strip().lower() or None
            content = bracket_match.group(4).strip()
        elif colon_idx != -1:
            raw_speaker_part = line_clean[:colon_idx].strip()
            content = line_clean[colon_idx + 1:].strip()
            spk_emo_match = re.match(r"^(.*?)\s*(?:\(([^)]+)\)|\[([^\]]+)\])\s*$", raw_speaker_part)
            if spk_emo_match:
                speaker = spk_emo_match.group(1).strip()
                emotion_tag = (spk_emo_match.group(2) or spk_emo_match.group(3) or "").strip().lower() or None
            else:
                speaker = raw_speaker_part
            speaker = re.sub(r"[\*_]+", "", speaker).strip()

        if speaker and content is not None:
            speaker = re.sub(r"[\*_]+", "", speaker).strip()
            content = re.sub(r"^[\*_]+\s*", "", content).strip()
            inner_emotion = re.match(r"^(?:\(([^)]+)\)|\[([^\]]+)\])\s*:?\s*(.*)$", content)
            if inner_emotion:
                if not emotion_tag:
                    emotion_tag = (inner_emotion.group(1) or inner_emotion.group(2) or "").strip().lower()
                content = inner_emotion.group(3).strip()
            if (content.startswith('"') and content.endswith('"')) or (content.startswith("'") and content.endswith("'")):
                if len(content) >= 2:
                    content = content[1:-1].strip()
            if content:
                turns.append(DialogueTurn(speaker=speaker, emotion_tag=emotion_tag, text=content, pause_after=default_pause))
        else:
            if turns:
                cleaned_cont = line_clean.strip("\"' ")
                if cleaned_cont:
                    turns[-1].text += f" {cleaned_cont}"
    return turns


def build_dialogue_subtitles(turns_metadata: List[dict]) -> Tuple[str, str]:
    """Membuat file subtitle SRT dan ASS Karaoke dengan label pembicara."""
    srt_lines: List[str] = []
    ass_events: List[str] = []

    ass_header = (
        "[Script Info]\n"
        "Title: intiVoice Dual-Speaker Dialogue Subtitles\n"
        "ScriptType: v4.00+\n"
        "WrapStyle: 0\n"
        "ScaledBorderAndShadow: yes\n"
        "YCbCr Matrix: None\n\n"
        "[V4+ Styles]\n"
        "Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding\n"
        "Style: Default,Arial,22,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,2,1,2,20,20,25,1\n\n"
        "[Events]\n"
        "Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text\n"
    )

    for idx, turn in enumerate(turns_metadata, start=1):
        speaker = turn.get("speaker", "Speaker")
        text = turn.get("text", "")
        start_sec = float(turn.get("start_sec", 0.0))
        end_sec = float(turn.get("end_sec", start_sec + 1.0))

        start_srt = format_timestamp_srt(start_sec)
        end_srt = format_timestamp_srt(end_sec)
        srt_lines.append(f"{idx}\n{start_srt} --> {end_srt}\n[{speaker}]: {text}\n")

        start_ass = format_timestamp_ass(start_sec)
        end_ass = format_timestamp_ass(end_sec)
        ass_events.append(f"Dialogue: 0,{start_ass},{end_ass},Default,,0,0,0,,{{\\b1}}[{speaker}]:{{\\b0}} {text}")

    srt_content = "\n".join(srt_lines)
    ass_content = ass_header + "\n".join(ass_events)
    return srt_content, ass_content

def assemble_dialogue_audio(turns_audio: List[np.ndarray], pauses_sec: List[float], sample_rate: int = 48000) -> np.ndarray:
    """Menyambungkan segmen-segmen audio percakapan dengan micro-fade 10ms dan jeda hening alami."""
    if not turns_audio:
        return np.zeros(0, dtype=np.float32)

    concatenated: List[np.ndarray] = []
    fade_len = int(sample_rate * 0.01) # 10ms fade

    for i, audio_seg in enumerate(turns_audio):
        if audio_seg is None or len(audio_seg) == 0:
            continue
        processed = apply_fade(audio_seg, fade_samples=fade_len)
        concatenated.append(processed)

        if i < len(turns_audio) - 1:
            p_dur = pauses_sec[i] if i < len(pauses_sec) else 0.40
            pause_samples = int(sample_rate * max(0.05, p_dur))
            concatenated.append(np.zeros(pause_samples, dtype=np.float32))

    if not concatenated:
        return np.zeros(0, dtype=np.float32)
    return np.concatenate(concatenated)

def _internal_synthesize_dialogue(req: DialogueGenerateRequest, on_progress=None) -> Dict[str, Any]:
    """Eksekusi GPU Colab untuk percakapan dua suara multi-speaker."""
    sample_rate = getattr(model.tts_model, "sample_rate", 48000)
    parsed_turns: List[DialogueTurn] = []
    if req.turns:
        parsed_turns = req.turns
    elif req.script:
        parsed_turns = parse_dialogue_script(req.script, default_pause=req.default_pause_sec or 0.40)

    if not parsed_turns:
        raise ValueError("Tidak ada giliran bicara (turns) yang valid dalam naskah.")

    temp_files: List[str] = []
    speaker_ref_cache: Dict[str, Optional[str]] = {}

    try:
        # 1. Pre-processing audio referensi kloning per pembicara
        for spk_name, spk_prof in req.speakers.items():
            ref_b64 = getattr(spk_prof, "reference_audio_b64", None)
            if ref_b64 and ref_b64.strip():
                try:
                    audio_bytes = base64.b64decode(ref_b64)
                    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_in:
                        tmp_in.write(audio_bytes)
                        raw_path = tmp_in.name
                        temp_files.append(raw_path)

                    ref_info = sf.info(raw_path)
                    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_16k:
                        ref_16k = tmp_16k.name
                        temp_files.append(ref_16k)

                    if ref_info.samplerate != 16000 or ref_info.channels > 1:
                        import torchaudio
                        waveform, orig_sr = torchaudio.load(raw_path)
                        if waveform.shape[0] > 1:
                            waveform = waveform.mean(dim=0, keepdim=True)
                        if orig_sr != 16000:
                            resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=16000)
                            waveform = resampler(waveform)
                        torchaudio.save(ref_16k, waveform, 16000)
                    else:
                        shutil.copyfile(raw_path, ref_16k)

                    speaker_ref_cache[spk_name] = ref_16k
                    # Dual-key: lowercase + canonical
                    speaker_ref_cache[spk_name.strip().lower()] = ref_16k
                    _canon = getattr(spk_prof, "name", None)
                    if _canon:
                        speaker_ref_cache[_canon] = ref_16k
                        speaker_ref_cache[_canon.strip().lower()] = ref_16k
                except Exception as e:
                    print(f"⚠️ Gagal memproses reference audio untuk {spk_name}: {e}")
                    speaker_ref_cache[spk_name] = None
            else:
                speaker_ref_cache[spk_name] = None

        turns_audio: List[np.ndarray] = []
        pauses_sec: List[float] = []
        turns_metadata: List[dict] = []
        current_time = 0.0
        total_turns = len(parsed_turns)

        # 2. Iterasi per giliran bicara
        for idx, turn in enumerate(parsed_turns):
            spk_name = turn.speaker
            spk_prof = speakers.get(spk_name)
            if not spk_prof:
                clean_target = spk_name.strip().lower()
                for k, v in speakers.items():
                    if k.strip().lower() == clean_target:
                        spk_prof = v
                        break
            if not spk_prof:
                spk_prof = next(iter(speakers.values())) if speakers else SpeakerProfile(name=spk_name)

            if on_progress:
                on_progress(f"Merender percakapan: giliran {idx + 1}/{total_turns} ({spk_name})...")

            # Kunci Seed pembicara
            turn_seed = getattr(spk_prof, "seed", None) or 42
            torch.manual_seed(turn_seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(turn_seed)

            # Ekstrak gaya & emosi
            style_parts: List[str] = []
            if turn.emotion_tag:
                clean_emotion = EMOTION_THESAURUS.get(turn.emotion_tag.lower(), turn.emotion_tag)
                if clean_emotion:
                    style_parts.append(clean_emotion)
            elif getattr(spk_prof, "style_prompt", None):
                clean_style = re.sub(r"[()（）]", "", spk_prof.style_prompt).strip()
                if clean_style:
                    style_parts.append(clean_style)

            # Dukungan kecepatan suara native via prompt kontrol
            spk_speed = getattr(spk_prof, "speed", 1.0) or 1.0
            if spk_speed <= 0.85:
                style_parts.append("bicara lambat")
            elif spk_speed >= 1.25:
                style_parts.append("bicara cepat")

            control_instruction = ", ".join(style_parts) if style_parts else None
            prompt_text = f"({control_instruction}){turn.text}" if control_instruction else turn.text

            gen_kwargs = {
                "text": prompt_text,
                "cfg_value": req.cfg_value or 2.0,
                "inference_timesteps": req.inference_timesteps or 10,
                "normalize": True,
                "retry_badcase": True,
                "retry_badcase_max_times": 2,
            }
            # Resolusi Master Timbre: dual-key + Dynamic Voice Anchor fallback
            spk_canon_key = spk_name.strip().lower()
            ref_path = (
                speaker_ref_cache.get(spk_name)
                or speaker_ref_cache.get(spk_name.strip().lower())
                or speaker_anchors.get(spk_canon_key)
            )
            if ref_path:
                gen_kwargs["reference_wav_path"] = ref_path

            with torch.inference_mode():
                wav = model.generate(**gen_kwargs)

            dur_turn = len(wav) / sample_rate
            turns_audio.append(wav)

            # DYNAMIC VOICE ANCHOR: Potong 3 detik pertama sebagai anchor giliran berikutnya
            spk_canon_key = spk_name.strip().lower()
            if (not speaker_ref_cache.get(spk_canon_key)) and (spk_canon_key not in speaker_anchors) and len(wav) > 0:
                try:
                    anchor_file = tempfile.NamedTemporaryFile(suffix=f"_anchor_{spk_canon_key}.wav", delete=False)
                    anchor_samples = min(len(wav), int(sample_rate * 3.0))
                    sf.write(anchor_file.name, wav[:anchor_samples], sample_rate)
                    speaker_anchors[spk_canon_key] = anchor_file.name
                    temp_files.append(anchor_file.name)
                except Exception as anc_err:
                    print(f"Warning: Gagal membuat Dynamic Voice Anchor untuk {spk_name}: {anc_err}")

            p_after = turn.pause_after if turn.pause_after is not None else (req.default_pause_sec or 0.40)
            pauses_sec.append(p_after)

            turns_metadata.append({
                "speaker": spk_name,
                "text": turn.text,
                "start_sec": current_time,
                "end_sec": current_time + dur_turn,
            })
            current_time += dur_turn + p_after

        # 3. Rakit Master Audio & Subtitle
        assembled_wav = assemble_dialogue_audio(turns_audio, pauses_sec, sample_rate=sample_rate)
        srt_content, ass_content = build_dialogue_subtitles(turns_metadata)
        total_duration = len(assembled_wav) / sample_rate if len(assembled_wav) > 0 else 0.0

        buf = io.BytesIO()
        sf.write(buf, assembled_wav, sample_rate, format="WAV")
        wav_bytes = buf.getvalue()

        # Simpan jika ada audio_id
        if req.audio_id:
            try:
                out_p = os.path.join(DRIVE_OUTPUT, f"{req.audio_id}.wav")
                with open(out_p, "wb") as f_out:
                    f_out.write(wav_bytes)
            except Exception:
                pass

        return {
            "wav_bytes": wav_bytes,
            "audio_b64": base64.b64encode(wav_bytes).decode("utf-8"),
            "duration": round(total_duration, 2),
            "sample_rate": sample_rate,
            "turns_count": len(parsed_turns),
            "srt": srt_content,
            "ass": ass_content,
            "engine": "colab",
        }

    finally:
        for tf in temp_files:
            try:
                if os.path.exists(tf):
                    os.remove(tf)
            except Exception:
                pass


# ------------------------------------------------------------------------------
# ENDPOINT PARITAS V1 & OPENAI COMPATIBLE
# ------------------------------------------------------------------------------
@api_app.post("/v1/tts")
def v1_tts(req: TTSRequest):
    """POST /v1/tts — Text-to-Speech sederhana langsung."""
    res = _internal_synthesize(req)
    return Response(content=res["wav_bytes"], media_type="audio/wav")

@api_app.post("/v1/design")
def v1_design(req: DesignRequest):
    """POST /v1/design — Voice Design dari deskripsi teks."""
    universal_req = TTSRequest(
        text=req.text,
        prompt_voice=req.voice_description,
        cfg_value=req.cfg_value,
        inference_timesteps=req.inference_timesteps,
        seed=req.seed,
        speed=req.speed
    )
    res = _internal_synthesize(universal_req)
    return Response(content=res["wav_bytes"], media_type="audio/wav")

@api_app.post("/v1/clone")
def v1_clone(req: CloneRequest):
    """POST /v1/clone — Voice Cloning cepat via Base64."""
    universal_req = TTSRequest(
        text=req.text,
        reference_audio_b64=req.reference_audio_b64,
        prompt_text=req.prompt_text,
        cfg_value=req.cfg_value,
        inference_timesteps=req.inference_timesteps,
        seed=req.seed,
        speed=req.speed
    )
    res = _internal_synthesize(universal_req)
    return Response(content=res["wav_bytes"], media_type="audio/wav")

@api_app.post("/v1/audio/speech")
def v1_openai_speech(req: OpenAIRequest):
    """POST /v1/audio/speech — OpenAI-compatible TTS endpoint."""
    universal_req = TTSRequest(
        text=req.input,
        prompt_voice=req.voice or "",
        speed=req.speed or 1.0
    )
    res = _internal_synthesize(universal_req)
    return Response(content=res["wav_bytes"], media_type="audio/wav")


@api_app.get("/outputs/check/{filename}")
def check_output(filename: str):
    """GET /outputs/check/{filename} — Periksa apakah file audio ada di penyimpanan Drive/Colab."""
    clean_name = filename if "." in filename else f"{filename}.wav"
    candidates = [
        os.path.join(DRIVE_OUTPUT, clean_name),
        os.path.join("/content/outputs", clean_name),
        os.path.join("/content/drive/MyDrive/intiVoice_Audio", clean_name)
    ]
    found = any(os.path.exists(p) for p in candidates)
    return {"exists": found, "filename": clean_name}

@api_app.get("/outputs/download/{filename}")
def download_output(filename: str):
    """GET /outputs/download/{filename} — Alirkan file audio WAV atau arsip subtitle langsung ke browser."""
    clean_name = filename if "." in filename else f"{filename}.wav"
    candidates = [
        os.path.join(DRIVE_OUTPUT, clean_name),
        os.path.join("/content/outputs", clean_name),
        os.path.join("/content/drive/MyDrive/intiVoice_Audio", clean_name)
    ]
    target_file = None
    for p in candidates:
        if os.path.exists(p):
            target_file = p
            break

    if not target_file:
        raise HTTPException(status_code=404, detail=f"Berkas '{clean_name}' tidak ditemukan di penyimpanan server.")

    media_type = "audio/wav"
    if clean_name.endswith(".srt") or clean_name.endswith(".txt") or clean_name.endswith(".ass"):
        media_type = "text/plain; charset=utf-8"

    with open(target_file, "rb") as f:
        return Response(
            content=f.read(),
            media_type=media_type,
            headers={
                "Content-Disposition": f'inline; filename="{clean_name}"',
                "Access-Control-Allow-Origin": "*",
                "Cache-Control": "public, max-age=3600"
            }
        )

# ------------------------------------------------------------------------------
# 9. MENJALANKAN SERVER FASTAPI & CLOUDFLARE TUNNEL (DUAL MODE)
# ------------------------------------------------------------------------------

# ------------------------------------------------------------------------------
# ENDPOINTS PERCAKAPAN DUA SUARA (DUAL-SPEAKER DIALOGUE v1.5.0)
# ------------------------------------------------------------------------------
@api_app.post("/v1/dialogue")
def v1_dialogue(req: DialogueGenerateRequest):
    """POST /v1/dialogue — Sintesis percakapan multi-speaker langsung (direct stream)."""
    res = _internal_synthesize_dialogue(req)
    return Response(
        content=res["wav_bytes"],
        media_type="audio/wav",
        headers={
            "X-Dialogue-Turns": str(res["turns_count"]),
            "X-Audio-Duration": str(res["duration"]),
            "X-Engine-Version": "1.5.0",
        }
    )

@api_app.post("/job/dialogue/create")
def job_dialogue_create(req: DialogueGenerateRequest):
    """POST /job/dialogue/create — Sintesis percakapan asinkron dengan progress tracking."""
    job_id = f"job_dlg_{int(time.time()*1000)}_{random.randint(1000,9999)}"
    JOB_REGISTRY[job_id] = {
        "status": "processing",
        "progress": "Memulai orkestrasi dialog multi-speaker...",
        "created_at": time.time(),
    }

    def _run_bg():
        try:
            def _prog(msg):
                if job_id in JOB_REGISTRY:
                    JOB_REGISTRY[job_id]["progress"] = msg

            res = _internal_synthesize_dialogue(req, on_progress=_prog)
            JOB_REGISTRY[job_id] = {
                "status": "completed",
                "progress": "Selesai 100%",
                "result": res,
                "completed_at": time.time(),
            }
        except Exception as e:
            JOB_REGISTRY[job_id] = {
                "status": "failed",
                "progress": "Gagal",
                "error": str(e),
                "failed_at": time.time(),
            }

    threading.Thread(target=_run_bg, daemon=True).start()
    return {"job_id": job_id, "status": "processing"}


def run_fastapi():
    import uvicorn
    uvicorn.run(api_app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=run_fastapi, daemon=True)
server_thread.start()
time.sleep(2)

print("\n" + "=" * 80)
print("🌐 MEMBUKA CLOUDFLARE TUNNEL KE INTERNET PUBLIK...")
print("=" * 80)

# 1. Cek token & kredensial dari form notebook
raw_token = CLOUDFLARE_TUNNEL_TOKEN if 'CLOUDFLARE_TUNNEL_TOKEN' in globals() and CLOUDFLARE_TUNNEL_TOKEN else ""
token_clean = raw_token.strip() if isinstance(raw_token, str) else ""

raw_domain = CLOUDFLARE_TUNNEL_DOMAIN if 'CLOUDFLARE_TUNNEL_DOMAIN' in globals() and CLOUDFLARE_TUNNEL_DOMAIN else ""
custom_domain = raw_domain.strip().lower().replace("https://", "").replace("http://", "").strip("/") if isinstance(raw_domain, str) else ""

raw_hf = HF_TOKEN if 'HF_TOKEN' in globals() and HF_TOKEN else ""
hf_token_clean = raw_hf.strip() if isinstance(raw_hf, str) else ""
if hf_token_clean:
    hf_token_clean = re.sub(r"^\s*HF_TOKEN\s*=\s*", "", hf_token_clean, flags=re.IGNORECASE).strip().strip('"').strip("'")

# AUTO-SANITIZER: Bersihkan prefix jika pengguna menempelkan 'CLOUDFLARE_TUNNEL_TOKEN=...' atau tanda kutip
if token_clean:
    token_clean = re.sub(r"^\s*CLOUDFLARE_TUNNEL_TOKEN\s*=\s*", "", token_clean, flags=re.IGNORECASE)
    token_clean = token_clean.strip().strip('"').strip("'")

# JIKA TOKEN DIISI DI FORM & GOOGLE DRIVE AKTIF -> AUTO-SAVE KE .ENV
if (token_clean or custom_domain or hf_token_clean) and DRIVE_MOUNTED:
    try:
        os.makedirs(DRIVE_APP_DIR, exist_ok=True)
        drive_env_target = f"{DRIVE_APP_DIR}/.env"
        # Baca isi lama agar tidak menimpa setting lain jika ada
        existing_env = {}
        if os.path.exists(drive_env_target):
            try:
                with open(drive_env_target, "r", encoding="utf-8") as fe:
                    for el in fe:
                        if "=" in el:
                            k, v = el.strip().split("=", 1)
                            existing_env[k.strip()] = v.strip()
            except Exception:
                pass
        
        if token_clean:
            existing_env["CLOUDFLARE_TUNNEL_TOKEN"] = token_clean
        if custom_domain:
            existing_env["CLOUDFLARE_TUNNEL_DOMAIN"] = custom_domain
        if hf_token_clean:
            existing_env["HF_TOKEN"] = hf_token_clean

        with open(drive_env_target, "w", encoding="utf-8") as f_save:
            for ek, ev in existing_env.items():
                f_save.write(f"{ek}={ev}\n")
        print(f"💾 Konfigurasi lisensi berhasil disimpan permanen ke Google Drive!")
        print(f"   📂 Lokasi File: {drive_env_target}")
        print(f"   💡 Sesi berikutnya, Anda tidak perlu mengisi token lagi (bisa dibiarkan kosong).")
    except Exception as e_save:
        print(f"⚠️ Gagal menyimpan .env ke Drive: {e_save}")

# 2. MULTI-PATH FALLBACK: BACA DARI GOOGLE DRIVE JIKA FORM KOSONG
if not token_clean and DRIVE_MOUNTED:
    drive_candidates = [
        f"{DRIVE_APP_DIR}/.env",
        "/content/drive/MyDrive/intiVoice_Audio/.env",
        "/content/drive/MyDrive/intiVoice_Studio/.env",
        "/content/drive/MyDrive/.env"
    ]
    for env_cand in drive_candidates:
        if os.path.exists(env_cand):
            try:
                with open(env_cand, "r", encoding="utf-8") as f_env:
                    for env_line in f_env:
                        env_line = env_line.strip()
                        if env_line.startswith("CLOUDFLARE_TUNNEL_TOKEN="):
                            token_clean = env_line.split("=", 1)[1].strip().strip('"').strip("'")
                        elif env_line.startswith("CLOUDFLARE_TUNNEL_DOMAIN="):
                            if not custom_domain:
                                custom_domain = env_line.split("=", 1)[1].strip().strip('"').strip("'")
                        elif env_line.startswith("HF_TOKEN="):
                            if not hf_token_clean:
                                hf_token_clean = env_line.split("=", 1)[1].strip().strip('"').strip("'")
                if token_clean:
                    print(f"📁 Terdeteksi konfigurasi lisensi dari Google Drive: {env_cand}")
                    if custom_domain:
                        print(f"   Domain Terdaftar: https://{custom_domain}/")
                    break
            except Exception as e_env:
                print(f"⚠️ Gagal membaca {env_cand}: {e_env}")

# 3. AUTO-DETECT DARI GOOGLE COLAB SECRETS
try:
    from google.colab import userdata
    if not token_clean:
        sec_token = userdata.get("CLOUDFLARE_TUNNEL_TOKEN")
        if sec_token:
            token_clean = sec_token.strip().strip('"').strip("'")
            print("🔑 Terdeteksi CLOUDFLARE_TUNNEL_TOKEN dari Google Colab Secrets.")
    if not hf_token_clean:
        sec_hf = userdata.get("HF_TOKEN")
        if sec_hf:
            hf_token_clean = sec_hf.strip().strip('"').strip("'")
            print("🤗 Terdeteksi HF_TOKEN dari Google Colab Secrets.")
except Exception:
    pass

# AKTIFKAN HF_TOKEN JIKA TERSEDIA
if hf_token_clean:
    os.environ['HF_TOKEN'] = hf_token_clean
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token_clean
    print("🚀 Akun Hugging Face Terautentikasi (Kecepatan Download Maksimal Aktif!)")

if token_clean:
    print("🔒 [MODE PERMANEN] Menjalankan Cloudflare Named Tunnel dengan Token...")
    tunnel_cmd = ["cloudflared", "tunnel", "run", "--token", token_clean]
    tunnel_process = subprocess.Popen(tunnel_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(3)
    if tunnel_process.poll() is not None:
        err_out = tunnel_process.stdout.read() if tunnel_process.stdout else ""
        print("❌ GAGAL MENJALANKAN TUNNEL PERMANEN! Output error:")
        print(err_out[:500])
        print("👉 Beralih otomatis ke Mode Quick Tunnel...")
        token_clean = ""

if token_clean:
    print("\n" + "🎉" * 40)
    print("🚀 MESIN INTIVOICE AI SUDAH ONLINE DENGAN URL PERMANEN!")
    print("=" * 80)
    target_url = f"https://{custom_domain}/" if custom_domain else "sesuai hostname Cloudflare Zero Trust Anda"
    print(f"👉 URL TUNNEL TETAP ANDA: {target_url}")
    print("=" * 80)
else:
    print("🌐 [MODE QUICK TUNNEL] Membuka URL sementara trycloudflare.com...")
    tunnel_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    tunnel_url = None
    start_tunnel_wait = time.time()

    while time.time() - start_tunnel_wait < 30:
        line = tunnel_process.stdout.readline()
        if not line:
            continue
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if match:
            tunnel_url = match.group(0)
            break

    if tunnel_url:
        print("\n" + "🎉" * 40)
        print("🚀 MESIN INTIVOICE AI SUDAH ONLINE DAN SIAP DIGUNAKAN!")
        print("=" * 80)
        print("👉 SALIN URL INI KE WEB STUDIO (Tab Pengaturan Mesin):")
        print(f"   {tunnel_url}")
        print("=" * 80)
    else:
        print("⚠️ Tunnel tidak merespons dalam 30 detik. Silakan periksa koneksi.")
